In [0]:

# 1. Force uninstall the breaking version
%pip uninstall -y meteostat

# 2. Install the specific compatible version and other dependencies
%pip install meteostat<2.0.0 geopy workalendar holidays xgboost
#%pip install pandas meteostat==1.6.5 workalendar holidays geopy xgboost


# 3. Force Databricks to restart the Python state and apply changes
dbutils.library.restartPython()


Found existing installation: meteostat 1.6.5
Uninstalling meteostat-1.6.5:
  Successfully uninstalled meteostat-1.6.5
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#Imports
import pandas as pd
import numpy as np
from datetime import datetime
from geopy.geocoders import Nominatim
from meteostat import Point, Daily
from workalendar.america.brazil import BrazilSaoPauloCity
import holidays
from pyspark.sql.functions import col, udf, to_date
from pyspark.sql.types import IntegerType
from pyspark.sql import SparkSession

In [0]:
#Spark Session
spark = SparkSession.builder.appName("Energy_Data_Ingestion_ETL").getOrCreate()

In [0]:
def get_energy_data(start_year, spark_session):
    """
    Extract energy consumption data from ONS Open Data API dynamically up to the current available data.
    """
    current_year = datetime.now().year
    dfs = []
    cols_to_keep = ['id_subsistema', 'nom_subsistema', 'din_instante', 'val_cargaenergiamwmed']
    
    for year in range(start_year, current_year + 1):
        url = f"https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_{year}.csv"
        print(f"Downloading energy data for {year}...")
        try:
            df = pd.read_csv(url, sep=";", encoding="utf-8")
            # Only append the required columns to guarantee schema alignment
           # Converte cols_to_keep em um conjunto (set) e verifica se todas essas colunas existem em df.columns
            if set(cols_to_keep).issubset(df.columns):
                dfs.append(df[cols_to_keep])
            else:
                print(f"Skipping {year} due to missing required columns.")
        except Exception as e:
            print(f"Could not download data for {year}. Skipping... (Error: {e})")
            continue

    if not dfs:
        raise ValueError("No energy data downloaded.")

    # Concat Df to DFs
    pdf_energy_raw = pd.concat(dfs, ignore_index=True)
    
    # Determine the absolute max date available in the ONS payload
    max_date_str = pdf_energy_raw['din_instante'].max()
    max_date = pd.to_datetime(max_date_str).to_pydatetime()
    
    # Force exact data types to prevent PyArrow ChunkedArray and Schema inference failures
    pdf_energy_raw['id_subsistema'] = pdf_energy_raw['id_subsistema'].astype(str)
    pdf_energy_raw['nom_subsistema'] = pdf_energy_raw['nom_subsistema'].astype(str)
    pdf_energy_raw['din_instante'] = pdf_energy_raw['din_instante'].astype(str)
    pdf_energy_raw['val_cargaenergiamwmed'] = pd.to_numeric(pdf_energy_raw['val_cargaenergiamwmed'], errors='coerce')
    
    # Flatten strictly to Python native lists to permanently bypass PyArrow RecordBatch ChunkedArray errors
    # Converte o DataFrame para listas nativas do Python (substituindo NaN por None)
    # para evitar erros do PyArrow ao lidar com estruturas internas como ChunkedArray
    cols = list(pdf_energy_raw.columns)
    data_list = pdf_energy_raw.replace({np.nan: None}).values.tolist()
    
    return spark_session.createDataFrame(data_list, schema=cols), max_date

In [0]:
def get_weather_data(start_date, end_date, spark_session):
    """
    Extract weather data for Sao Paulo from Meteostat API.
    """
    print("Downloading weather data...")
    geolocator = Nominatim(user_agent="energy-Load-Prediction")
    loc = geolocator.geocode("São Paulo, SP, Brazil")
    sp = Point(loc.latitude, loc.longitude)

    # Fetch data
    df_weather_pd = Daily(sp, start_date, end_date).fetch().reset_index()
    
    # Rename and select columns
    df_weather_pd = df_weather_pd.rename(columns={"time": "date", "tavg": "temp_mean_c"})
    df_weather_pd = df_weather_pd[["date", "temp_mean_c"]]
    
    # Protect against PySpark STRUCT inference over native objects
    df_weather_pd["date"] = df_weather_pd["date"].astype(str)
    
    # Flatten strictly to Python native lists to permanently bypass PyArrow RecordBatch ChunkedArray errors
    cols = list(df_weather_pd.columns)
    data_list = df_weather_pd.replace({np.nan: None}).values.tolist()
    
    return spark_session.createDataFrame(data_list, schema=cols)

In [0]:
def transform_energy_data(df_energy):
    """
    Clean and apply business rules to energy data.
    """
    # Filter for 'SE' (Sudeste/Centro-Oeste)
    df_se = df_energy.filter(col('id_subsistema') == 'SE')
    
    # Drop unnecessary columns
    df_se = df_se.drop('id_subsistema', 'nom_subsistema')
    
    # Rename columns
    df_se = df_se.withColumnRenamed('din_instante', 'date') \
                 .withColumnRenamed('val_cargaenergiamwmed', 'energy_load_MWmed')

    # Apply business rule: consumption is exactly 0.05% of the total verified load, multiplied by 24 to get daily load
    df_se = df_se.withColumn('consumption_mwh_day', col('energy_load_MWmed') * 0.0005 * 24)
    df_se = df_se.drop('energy_load_MWmed')
    
    # Convert date string to DateType
    return df_se.withColumn('date', to_date(col('date')))

In [0]:
def transform_weather_data(df_weather):
    """
    Format Datetime schema for weather.
    """
    return df_weather.withColumn('date', to_date(col('date')))

In [0]:
def add_holiday_flag(df):
    """
    Add boolean flag indicating if the date is a holiday in Sao Paulo.
    """
    cal = BrazilSaoPauloCity()

    def get_is_holiday(date_val):
        if date_val is None:
            return None
        return int(cal.is_holiday(date_val))

    # Register UDF
    holiday_udf = udf(get_is_holiday, IntegerType())
    
    return df.withColumn("is_holiday", holiday_udf(col("date")))

In [0]:
def merge_and_finalize(df_energy, df_weather):
    """
    Merge datasets and finalize structure.
    """
    # Merge Daily Energy and average temp DataFrames
    df_merged = df_energy.join(df_weather, on="date", how="left")
    
    # Add holiday flag
    df_final = add_holiday_flag(df_merged)
    
    # Sort by date
    return df_final.orderBy("date")

In [0]:
def save_to_lakehouse(df, table_name):
    """
    Save the DataFrame as a Delta Table in Databricks.
    """
    print(f"Saving data to delta table: {table_name}")
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)

#ETL

In [0]:
# 1. Pipeline parameters (Updated to match recent changes)
ENERGY_START_YEAR = 2019
WEATHER_START = datetime(2019, 1, 1)
TARGET_TABLE = "foodshop_consumption"



In [0]:
# 2. Extract
print("Starting Extract Phase...")
df_energy_raw, weather_end_date = get_energy_data(ENERGY_START_YEAR, spark)
print(f"Latest ONS record bounds the dataset to: {weather_end_date.strftime('%Y-%m-%d')}")

df_weather_raw = get_weather_data(WEATHER_START, weather_end_date, spark)



In [0]:
# 3. Transform
print("Starting Transform Phase...")
df_energy_clean = transform_energy_data(df_energy_raw)
df_weather_clean = transform_weather_data(df_weather_raw)
df_final = merge_and_finalize(df_energy_clean, df_weather_clean)



In [0]:
# Display a sample of the transformed data
#print("Transformation completed. Sample of final data:")
#df_final.display() if 'display' in globals() else df_final.show(5)



In [0]:
# 4. Load
print("Starting Load Phase...")
save_to_lakehouse(df_final, TARGET_TABLE)
print("ETL Pipeline completed successfully!")